In [ ]:
import glob

!fast-plate-ocr export \
  --format onnx \
  --plate-config-file ./config/cct_s_v2_global_plate_config.yaml \
  --simplify \
  --model {glob.glob("weights/*/best.keras")[0]}

In [ ]:
import glob
from fast_plate_ocr import LicensePlateRecognizer
from PIL import Image
from tqdm import tqdm

model_path = glob.glob("weights/*/best.onnx")[0]
model = LicensePlateRecognizer(
    onnx_model_path=model_path,
    plate_config_path="./config/cct_s_v2_global_plate_config.yaml",
)

images_path = glob.glob("./dataset/lp_crops_balanced/val/*.jpg")

for image_path in images_path[:5]:
    with open(image_path.replace(".jpg", ".txt")) as f:
        gt = f.read().strip()
    image = Image.open(image_path)
    label = model.run(image_path)[0].plate
    display(image)
    print("gt:", gt)
    print("pr:", label, end="\n\n")
    break

In [ ]:
import os
import math
import glob
import random
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
from tqdm import tqdm

def plot_predictions(
    images_path,
    infer_fn,
    n=12,
    cols=4,
    figsize=(16, 10),
    border_size=4,
    random_sample=True
):
    # Do NOT modify original list
    if random_sample:
        sampled_paths = random.sample(images_path, min(n, len(images_path)))
    else:
        sampled_paths = images_path[:n]

    rows = math.ceil(len(sampled_paths) / cols)

    plt.figure(figsize=figsize, facecolor="black")

    for idx, image_path in enumerate(tqdm(sampled_paths)):
        # Load GT
        label_path = image_path.replace(".jpg", ".txt")
        gt = ""
        if os.path.exists(label_path):
            with open(label_path) as f:
                gt = f.read().strip()

        # Load image
        image = Image.open(image_path)

        # Prediction
        pred = infer_fn(image_path)[0].plate

        # Normalize
        gt_norm = gt.strip().upper()
        pred_norm = pred.strip().upper()

        # Distance
        dist = levenshtein_distance(gt_norm, pred_norm)

        # Border color
        if pred_norm == gt_norm:
            color = "green"
        elif dist == 1:
            color = "yellow"
        else:
            color = "red"

        # Add border
        image_with_border = ImageOps.expand(
            image,
            border=border_size,
            fill=color
        )

        # Plot
        ax = plt.subplot(rows, cols, idx + 1)
        ax.set_facecolor("black")
        ax.imshow(image_with_border)
        ax.axis("off")

        ax.set_title(
            f"GT: {gt}\nPR: {pred}",
            fontsize=9,
            color="white"
        )

    plt.tight_layout()
    # plt.savefig("predicted.jpg", facecolor="black", bbox_inches="tight")
    plt.show()

images_path = glob.glob("./dataset/lp_crops_balanced/train/*.jpg")

plot_predictions(
    images_path,
    infer_fn=model.run,
    n=100,
    cols=10,
    border_size=8,
    random_sample=True
)

In [ ]:
def levenshtein_distance(s1, s2):
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)

    if len(s2) == 0:
        return len(s1)

    prev_row = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = prev_row[j + 1] + 1
            deletions = curr_row[j] + 1
            substitutions = prev_row[j] + (c1 != c2)
            curr_row.append(min(insertions, deletions, substitutions))
        prev_row = curr_row

    return prev_row[-1]

total = 0
d0 = 0   # exact
d1 = 0   # <=1
d2 = 0   # <=2

images_path = glob.glob("./dataset/lp_crops_balanced/val/*.jpg")

for image_path in tqdm(images_path):
    with open(image_path.replace(".jpg", ".txt")) as f:
        gt = f.read().strip()
        if ("\"" in gt) or (" " in gt):
            continue

    image = Image.open(image_path)
    pred = model.run(image_path)[0].plate

    # dist = levenshtein_distance(gt, pred)
    dist = levenshtein_distance(gt, pred)

    total += 1

    if dist == 0:
        d0 += 1
    if dist <= 1:
        d1 += 1
    if dist <= 2:
        d2 += 1

acc_exact = d0 / total
acc_1 = d1 / total
acc_2 = d2 / total

print(f"Exact accuracy (distance=0): {acc_exact:.4f}")
print(f"Accuracy (distance<=1):     {acc_1:.4f}")
print(f"Accuracy (distance<=2):     {acc_2:.4f}")